In [2]:
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator
from ta.trend import EMAIndicator, SMAIndicator

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest


from datetime import timedelta

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [3]:
# swing structure direct Bullish to Bearish
def implement_market_structure_states_direct(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = None

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH" or current_state is None:
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "BEARISH"

        elif current_state == "BEARISH" or current_state is None:
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "BULLISH"

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            higher_low = last_confirmed_low


    df["market_state"] = states
    return df


In [4]:
# define resistant level
def define_resistant_level(df, define_key_level_window = 20):
    # gap
    gap = 0.3

    # all resistant highs
    resistant_list = []
    
    for row in df.iterrows():
        start_index = row[0] - timedelta(days=define_key_level_window)
        back_candle_range = df.loc[start_index:row[0]]
        pivot_lows = back_candle_range[back_candle_range['is_swing_low'] == True]

        for candle in pivot_lows.iterrows():
            frequency = 0
            total_candle = 0
            # resistant highs related to current high
            resistant_zone = []

            for candle2 in pivot_lows.iterrows():
                # candle 2 date needs to be before candle date
                if (candle2[0] <= candle[0] and 
                    candle[1]['Low'] + gap > candle2[1]['Low'] and 
                    candle[1]['Low'] - gap < candle2[1]['Low']):
                    frequency += 1
                    total_candle += candle2[1]['Low']
                    # resistant_zone.append(candle2)

            if frequency > 2:
                mean_candle = round(total_candle / frequency, 3)
                resistant_zone.append(candle)

                # for all new resistant, check if the candle is in resistant list, if not add it
                for i in range(len(resistant_zone)):
                    is_resistant = False

                    for j in range(len(resistant_list)):
                        if resistant_zone[i][0] == resistant_list[j][0][0]:
                            # print(f"{resistant_zone[i][0]} = {resistant_list[j][0][0]}")
                            is_resistant = True

                    if not is_resistant:
                        # print(f"{resistant_zone[i][0]} mean_candle = {mean_candle} & frequency = {frequency}")
                        resistant_candle = [resistant_zone[i], mean_candle]
                        resistant_list.append(resistant_candle)

    # print(resistant_list)
    return resistant_list


In [5]:
pair = 'GBPJPY'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.

In [6]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'
# end_date = '2026-9-16 00:00'

timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 15

# window for look back for key level
back_candles_window = 20

# gap t avoid look ahead
gap_look_ahead = 1

# nombre de candle au dessus de key level après breakou
breakout_candles_below = 9

# nombre de jours à chercher key level
key_level_lookback_days = 25

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles_below} candles au dessus du resistant level
breakout_candles_below_lookback = 30


In [7]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = implement_market_structure_states_direct(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
print(df_d1[df_d1["state_changed"]][["Close", "market_state"]])
df_d1 = df_d1.loc[start_date : end_date]


              Close market_state
Date                            
2020-12-02  139.632          NaN
2020-12-03  139.680          NaN
2020-12-04  139.937          NaN
2020-12-07  139.114          NaN
2020-12-08  139.134          NaN
...             ...          ...
2026-03-30  210.598      BEARISH
2026-04-09  213.599      BULLISH
2026-04-30  213.107      BEARISH
2026-05-25  214.592      BULLISH
2026-07-30  215.005      BEARISH

[76 rows x 2 columns]


In [8]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = implement_market_structure_states_direct(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1
Date,,,,,,,,,
2021-01-04 00:00:00,140.944,141.304,140.776,141.005,37352,True,False,BULLISH,BEARISH
2021-01-04 04:00:00,141.007,141.043,140.847,140.938,20550,False,False,BULLISH,BEARISH
2021-01-04 08:00:00,140.936,141.064,140.412,140.757,47343,False,False,BULLISH,BEARISH
2021-01-04 12:00:00,140.758,140.861,140.188,140.226,55445,False,False,BULLISH,BEARISH
2021-01-04 16:00:00,140.227,140.233,139.687,139.870,55631,False,False,BULLISH,BEARISH
...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,208.652,209.090,208.514,208.655,56407,False,False,BEARISH,BEARISH
2026-09-15 12:00:00,208.655,209.211,208.607,209.168,74480,False,False,BULLISH,BEARISH
2026-09-15 16:00:00,209.167,209.224,208.885,209.066,38410,False,False,BULLISH,BEARISH


In [9]:
def swing_point_pos(x):
    if x['is_swing_low']==True:
        return x['Low']-1e-4
    elif x['is_swing_high']==True:
        return x['High']+1e-4
    else:
        return np.nan

df_h4['swing_point'] = df_h4.apply(lambda row: swing_point_pos(row), axis=1)

In [10]:
start_date_chart = '2025-1-1 00:00:00'
end_date_chart = '2025-12-30 00:00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=df_h4['Open'],
                             high=df_h4['High'],
                             low=df_h4['Low'],
                             close=df_h4['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['swing_point'], mode="markers",
                         marker=dict(size=6, color="black"),
                         name="swing point"),
              row=1, col=1)

fig.update_layout(width=1200, height=800, sliders=[])
fig.show()

In [11]:
# resistants_list = define_resistant_level(df_h4, define_key_level_window)
resistants_list = define_resistant_level(df_h4, define_key_level_window)
resistants_list

[[(Timestamp('2021-01-26 08:00:00'),
   Open                141.457
   High                 141.81
   Low                 141.277
   Close               141.727
   Tickvol               37726
   is_swing_high         False
   is_swing_low           True
   market_state        BEARISH
   market_state_d1     BULLISH
   swing_point        141.2769
   Name: 2021-01-26 08:00:00, dtype: object),
  141.347],
 [(Timestamp('2021-04-16 04:00:00'),
   Open                149.823
   High                149.839
   Low                 149.381
   Close               149.445
   Tickvol               16885
   is_swing_high         False
   is_swing_low           True
   market_state        BEARISH
   market_state_d1     BULLISH
   swing_point        149.3809
   Name: 2021-04-16 04:00:00, dtype: object),
  149.516],
 [(Timestamp('2021-05-07 12:00:00'),
   Open                151.854
   High                151.866
   Low                 151.261
   Close               151.762
   Tickvol               6959

In [12]:
def resistant_pos(resistants_list, candle):
    for resistant_pt in resistants_list:
        if candle.name == resistant_pt[0][0]:
            # print(candle)
            return resistant_pt[1]

    return np.nan

def is_resistant(resistants_list, candle):
    for resistant_pt in resistants_list:
        if candle.name == resistant_pt[0][0]:
            # print(candle)
            return True

    return False

df_h4['resistant_point'] = df_h4.apply(lambda row: resistant_pos(resistants_list, row), axis=1)
df_h4['is_resistant'] = df_h4.apply(lambda row: is_resistant(resistants_list, row), axis=1)

In [13]:
start_date_chart = '2022-11-1 00:00:00'
end_date_chart = '2022-12-30 00:00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['resistant_point'], mode="markers",
                         marker=dict(size=8, color="red"),
                         name="resistant point"),
              row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [14]:
# back candle is the number of day to look back for resistant
# number of day to skip in order to avoid look-ahead
def get_resistant_list(df, current_index, back_candles_window=20, gap=1):
    start_date = current_index - timedelta(days=back_candles_window)
    end_date = current_index - timedelta(days=gap)

    window = df.loc[start_date : end_date]
    resistants_level = window[window['is_resistant'] == True]
    return resistants_level.groupby('resistant_point')['resistant_point'].max()

# candle_index = datetime.strptime('2024-02-15 10:15', date_format)
# print(get_resistant_list(df_m15, candle_index, key_level_lookback_days, 1))

In [15]:
# detect if all candles are below resistant level
def is_candles_below_resistant_level_list(df, breakout_candles_below=14, key_level_lookback_days=25):
    valid_resistant_level_list = []

    for index in range(len(df)):
        candle = df_h4.iloc[index]
        level_list = []
        resistant_levels_list = get_resistant_list(df, candle.name, key_level_lookback_days, 1)

        for level in resistant_levels_list:
            if (df['High'].iloc[index-breakout_candles_below:index] < level).all():
                level_list.append(level)

        valid_resistant_level_list.append(level_list)

    return valid_resistant_level_list

df_h4['valid_resistant_levels'] = is_candles_below_resistant_level_list(df_h4, breakout_candles_below, key_level_lookback_days)

In [16]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = implement_market_structure_states_direct(df_m15, n=structure_m15_window)

m15_daily_direction_list = []
m15_h4_key_level_list = []
m15_is_resistants_list = []
m15_is_below_resistant_list = []

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        m15_h4_key_level_list.append(df_h4.iloc[i]['resistant_point'])
        m15_is_resistants_list.append(df_h4.iloc[i]['is_resistant'])
        m15_is_below_resistant_list.append(df_h4.iloc[i]['valid_resistant_levels'])
        # affect current market states of H4 candle because the result is market state of previous D1 candle
        m15_daily_direction_list.append(df_h4.iloc[i]['market_state_d1'])

df_m15['resistant_point'] = m15_h4_key_level_list
df_m15['is_resistant'] = m15_is_resistants_list
df_m15['valid_resistant_levels'] = m15_is_below_resistant_list
df_m15['market_state_d1'] = m15_daily_direction_list

indicator_atr_14 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_14._atr

df_m15

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,resistant_point,is_resistant,valid_resistant_levels,market_state_d1,ATR
Date,,,,,,,,,,,,,
2021-01-04 00:00:00,140.944,141.110,140.944,141.100,2767,False,True,NaN,NaN,False,[],BEARISH,0.000000
2021-01-04 00:15:00,141.099,141.204,141.070,141.186,1223,False,False,NaN,NaN,False,[],BEARISH,0.000000
2021-01-04 00:30:00,141.186,141.304,141.158,141.282,1042,True,False,NaN,NaN,False,[],BEARISH,0.000000
2021-01-04 00:45:00,141.278,141.301,141.218,141.244,1886,False,False,NaN,NaN,False,[],BEARISH,0.000000
2021-01-04 01:00:00,141.247,141.275,140.988,141.007,3577,False,False,NaN,NaN,False,[],BEARISH,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-15 23:00:00,208.895,209.029,208.895,209.018,1853,True,False,BEARISH,NaN,False,[],BEARISH,0.084410
2026-09-15 23:15:00,209.020,209.027,208.982,208.996,1268,False,False,BEARISH,NaN,False,[],BEARISH,0.081595
2026-09-15 23:30:00,208.997,209.013,208.926,208.953,1279,False,False,BEARISH,NaN,False,[],BEARISH,0.081981


In [17]:
candle_index = datetime.strptime('2022-11-15 06:00', date_format)
print(get_resistant_list(df_m15, candle_index, key_level_lookback_days, 1))

resistant_point
165.036    165.036
Name: resistant_point, dtype: float64


In [18]:
def get_valid_resistant_level(df, index, breakout_candles_below_lookback=14):
    periode = df.iloc[index-1-breakout_candles_below_lookback*16:index-1]
    valid_resistant_levels = []

    for candle in periode.iterrows():
        if len(candle[1]['valid_resistant_levels']) > 0:
            for i in range(len(candle[1]['valid_resistant_levels'])):
                valid_resistant_levels.append(candle[1]['valid_resistant_levels'][i])

    return list(set(valid_resistant_levels))

# 2024-03-25 06:00:00 2300
print(get_valid_resistant_level(df_m15, 2956, 30))

[]


In [19]:
# using m15 market structure for entry
def apply_total_signal(df, key_level_lookback_days = 20, gap=1, breakout_candles_below_lookback = 30):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    start_trade_hour = 5
    stop_trade_hour = 20
    current_day = None
    is_key_level_touched = False
    valid_resistant_levels = []
    is_below_resistant_levels = False

    high_tf_condition_short_satisfied = False
    high_tf_condition_short_candle = None

    # we want market structure to switch from bearish to bullish when htf conditions are met
    # if market structure is already bullish we need to wait
    previous_market_structure = None

    for i in range(0, len(df)):
        current_bar_date = df.index[i]
        signal = 0

        if current_day != current_bar_date.date() or current_bar_date.time().hour >= stop_trade_hour:
            current_day = current_bar_date.date()
            high_tf_condition_short_satisfied = False
            is_key_level_touched = False
            valid_resistant_levels = []
            is_below_resistant_levels = False

        # h4 key level conditions
        resistants_levels = get_resistant_list(df, current_bar_date, key_level_lookback_days, gap)
        resistants_levels_touched = []
        for level in resistants_levels:
            if df['High'].iloc[i-1] > level:
                # print(level)
                resistants_levels_touched.append(level)
                is_key_level_touched = True

        c_time_trade = start_trade_hour < current_bar_date.hour and current_bar_date.hour < stop_trade_hour

        if c_time_trade:
            # d1 trending conditions
            c_short_daily_direction = df['market_state_d1'].iloc[i-1] == 'BEARISH'
            c_m15_ms_bearish = df['market_state'].iloc[i] == 'BEARISH'

            # breakout_candles_below candle needs to be below resistant level for the breakout to be valid
            valid_resistant_levels = get_valid_resistant_level(df, i, breakout_candles_below_lookback)

            for valid_level in valid_resistant_levels:
                for level_touched in resistants_levels_touched:
                    if valid_level == level_touched:
                        # print(f"{current_bar_date} {valid_level} = {level_touched}")
                        # print(f"c_m15_ms_bullish = {c_m15_ms_bullish} & previous_market_structure = {previous_market_structure}")
                        is_below_resistant_levels = True

            # m15 market structure shift to bullish

            # print(current_bar_date)
            # Combine conditions
            if (c_short_daily_direction and
                is_key_level_touched and
                not high_tf_condition_short_satisfied):
                high_tf_condition_short_satisfied = True
                high_tf_condition_short_candle = current_bar_date
                previous_market_structure = df['market_state'].iloc[i]
                print(f"{current_bar_date} {i}")

            if (high_tf_condition_short_satisfied == True and
                c_m15_ms_bearish and
                high_tf_condition_short_candle < current_bar_date and
                is_below_resistant_levels and
                previous_market_structure == 'BULLISH'):
                # print(f"high_tf_condition_long_candle = {high_tf_condition_long_candle} & current_bar_date = {current_bar_date}")
                # print(f"resistants_levels_touched = {resistants_levels_touched}")
                # print(f"valid_resistant_levels = {valid_resistant_levels}")
                signal = 1

        if previous_market_structure != df['market_state'].iloc[i]:
            previous_market_structure = df['market_state'].iloc[i]    

        df_signal_list.append(signal)

    return df_signal_list

df_m15['TotalSignal'] = apply_total_signal(df=df_m15, key_level_lookback_days=key_level_lookback_days, gap=gap_look_ahead, breakout_candles_below_lookback=breakout_candles_below_lookback)

2021-06-10 06:00:00 10736
2021-06-11 06:00:00 10832
2021-06-14 06:00:00 10920
2021-06-15 06:00:00 11016
2021-06-16 06:00:00 11112
2021-06-17 06:00:00 11208
2021-06-18 06:00:00 11304
2021-06-23 06:00:00 11584
2021-06-24 06:00:00 11680
2021-06-25 06:00:00 11776
2021-07-28 08:00:00 13952
2021-07-29 06:00:00 14040
2021-07-30 06:00:00 14136
2021-08-02 06:00:00 14224
2021-08-03 06:00:00 14320
2021-08-04 06:00:00 14416
2021-08-05 06:00:00 14512
2021-08-06 06:00:00 14608
2021-08-09 06:00:00 14696
2021-08-10 06:00:00 14792
2021-08-11 06:00:00 14888
2021-08-12 06:00:00 14984
2021-08-13 06:00:00 15080
2021-08-16 06:00:00 15168
2021-09-27 06:00:00 18000
2021-11-10 16:00:00 21064
2021-11-11 06:00:00 21120
2021-11-12 07:30:00 21222
2021-11-15 06:00:00 21308
2021-11-16 06:00:00 21404
2021-12-10 12:00:00 23144
2021-12-13 06:00:00 23212
2021-12-14 06:00:00 23308
2021-12-15 06:00:00 23404
2022-07-13 12:00:00 37639
2022-07-14 06:00:00 37711
2022-07-15 06:00:00 37807
2022-07-18 06:00:00 37895
2022-08-01 0

In [20]:
len(df_m15[df_m15.TotalSignal != 0])

40

In [21]:
def pointpos(x):
    if x['TotalSignal']==2:
        return x['Low']-1e-4
    elif x['TotalSignal']==1:
        return x['High']+1e-4
    else:
        return np.nan

df_m15['pointpos'] = df_m15.apply(lambda row: pointpos(row), axis=1)

In [22]:
# Affichage des transitions d'états m15
df_m15["state_changed"] = df_m15["market_state"] != df_m15[
    "market_state"
].shift(1)
df_state_changed = df_m15[df_m15["state_changed"]][["Close", "market_state"]]

mk_bullish_pos_list = []
mk_bearish_pos_list = []

for i in range(len(df_m15)):
    previous_candle = df_m15.iloc[i-1]
    candle = df_m15.iloc[i]
    if candle['market_state'] == 'BULLISH' and previous_candle['market_state'] != 'BULLISH':
        mk_bullish_pos_list.append(candle['Low'] - 1e-4)
        mk_bearish_pos_list.append(np.nan)
        continue        

    if candle['market_state'] == 'BEARISH' and previous_candle['market_state'] != 'BEARISH':
        mk_bullish_pos_list.append(np.nan)
        mk_bearish_pos_list.append(candle['High'] + 1e-4)
        continue

    mk_bullish_pos_list.append(np.nan)
    mk_bearish_pos_list.append(np.nan)

df_m15['mk_bullish_pos'] = mk_bullish_pos_list
df_m15['mk_bearish_pos'] = mk_bearish_pos_list

In [23]:
start_date_chart = '2022-1-1 00:00:00'
end_date_chart = '2022-12-30 00:00:00'
dfpl = df_m15.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['support_point'], mode="markers",
#                          marker=dict(size=8, color="green"),
#                          name="support point"),
#               row=1, col=1)

# Add markers for entry point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

# # Add markers for market structure daily = bullish
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['mk_bullish_pos'], mode="markers",
#                          marker=dict(size=8, color="green"),
#                          name="bullish"),
#               row=1, col=1)

# # Add markers for market structure daily = bearish
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['mk_bearish_pos'], mode="markers",
#                          marker=dict(size=8, color="red"),
#                          name="bearish"),
#               row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [24]:
# dfopt = df_m15.loc['2022-07-1 00:00:00' : '2022-12-30 00:00:00']
dfopt = df_m15


In [25]:
# add some sl tp management
def SIGNAL():
    return dfopt.TotalSignal

class MyStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    last_swing_low = None
    last_swing_high = None
    previous_swing_low = None
    previous_swing_high = None
    previous_market_state = 'BULLISH'
    breakout_candles_below_lookback = 0
    key_level_gap = 0.2
    
    def init(self):
        self.current_day = None
        self.order_placed_time = None

        super().init()
        self.breakout_candles_below_lookback = breakout_candles_below_lookback
        self.signal1 = self.I(SIGNAL)


    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True

                if order_executed == False:
                    order.cancel()

        self.current_day = day
        self.order_placed_time = None

    def get_tag(self):
        return f"{self.order_placed_time}"
    
    def next(self):
        super().next()

        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date:
            self.reset_range(current_bar_date)

        if len(self.trades)==0:
            if self.signal1[-1]==1:
                valid_resistant_list = get_valid_resistant_level(dfopt, len(self.data.index), breakout_candles_below_lookback)
                if len(valid_resistant_list) > 0:
                    planned_entry_price = 0
                    sl = 0
                    if self.data.Close[-1] < max(valid_resistant_list) + self.key_level_gap: 
                        planned_entry_price = self.data.Close[-1]
                        sl = abs(self.data.Close[-1] - self.data.High[len(self.data.High) - 100 : len(self.data.High)].max())

                    else :
                        planned_entry_price = max(valid_resistant_list) + self.key_level_gap
                        sl = abs((max(valid_resistant_list) + self.key_level_gap) - self.data.High[len(self.data.High) - 100 : len(self.data.High)].max())
                    
                    tp = self.TPcoef * sl
    
                    position_size = int((self.risk_percent * self.equity) / sl)
    
                    sl1 = planned_entry_price + sl
                    tp1 = planned_entry_price - tp
                    # print(f"go long date= {self.data.index[-1]} & sl1 = {sl1} & tp1 = {tp1}")

                    if self.data.Close[-1] < max(valid_resistant_list) + self.key_level_gap: 
                        # print("Trade valid:")
                        # valid = input()
                        # print(f"trade = {valid}")
                        self.sell(sl=sl1, tp=tp1, size=position_size)
                        print(f"{self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]}")
                    else:
                        self.sell(sl=sl1, tp=tp1, size=position_size, stop=planned_entry_price)
                        print(f"{self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & planned_entry_price = {planned_entry_price}")

                    self.order_placed_time = self.data.index[-1]
                

In [26]:
bt = Backtest(dfopt, MyStrat, cash=10000, margin=0.02) #0.0002
stats = bt.run()

print(stats)

bt.plot()

2021-06-23 10:00:00 sl = 155.102 & sgnal = 1
2021-06-23 15:15:00 sl = 155.153 & sgnal = 1 & planned_entry_price = 154.819
2021-06-24 12:00:00 sl = 155.145 & sgnal = 1
2022-09-30 11:15:00 sl = 162.164 & sgnal = 1
2023-03-22 13:30:00 sl = 163.333 & sgnal = 1
2023-03-22 18:45:00 sl = 163.333 & sgnal = 1
2023-03-29 15:30:00 sl = 163.473 & sgnal = 1
2023-07-17 19:45:00 sl = 182.142 & sgnal = 1
2023-07-21 12:45:00 sl = 182.519 & sgnal = 1
2023-07-31 13:00:00 sl = 183.183 & sgnal = 1 & planned_entry_price = 182.315
2023-09-29 13:15:00 sl = 183.016 & sgnal = 1
2023-10-02 10:45:00 sl = 183.016 & sgnal = 1
2023-10-10 18:00:00 sl = 182.781 & sgnal = 1
2023-10-11 08:00:00 sl = 183.114 & sgnal = 1
2024-02-02 12:00:00 sl = 187.329 & sgnal = 1
2024-02-02 18:30:00 sl = 187.729 & sgnal = 1
2024-02-07 15:30:00 sl = 187.259 & sgnal = 1
2024-02-08 10:00:00 sl = 188.004 & sgnal = 1 & planned_entry_price = 187.451
2024-02-08 14:15:00 sl = 188.22 & sgnal = 1 & planned_entry_price = 187.451
2024-11-20 11:45:0

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1308', ...)